<a href="https://colab.research.google.com/github/winsonne/YBIGTA_newbie_assignment/blob/main/%EB%B9%85%EC%BD%98_%EA%B8%B0%ED%9B%84%EC%B6%A9%EA%B2%A9_%EC%B6%94%EA%B0%80%EB%B6%84%EC%84%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from pathlib import Path

Mounted at /content/drive


In [2]:
from pathlib import Path
import unicodedata
import pandas as pd
import numpy as np

def nfc(x):
    return unicodedata.normalize('NFC', str(x))

MYDRIVE = Path('/content/drive/MyDrive')

# 와빅_빅콘 폴더 자동 탐색
base_candidates = [
    p for p in MYDRIVE.iterdir()
    if p.is_dir()
    and nfc(p.name) == '와빅_빅콘'
]

print("와빅_빅콘 후보 수:", len(base_candidates))

BASE = base_candidates[0]

# climate_processed 폴더 자동 탐색
climate_candidates = [
    p for p in BASE.iterdir()
    if p.is_dir()
    and nfc(p.name) == 'climate_processed'
]

print("climate_processed 후보 수:", len(climate_candidates))

CLIMATE_DIR = climate_candidates[0]

print("실제 경로:", CLIMATE_DIR)

print("\n폴더 안 파일:")
for p in CLIMATE_DIR.iterdir():
    print(p.name)

와빅_빅콘 후보 수: 1
climate_processed 후보 수: 1
실제 경로: /content/drive/MyDrive/와빅_빅콘/climate_processed

폴더 안 파일:
README.txt
rainfall_daily.csv
rainfall_6h.csv


In [3]:
daily_candidates = [
    p for p in CLIMATE_DIR.iterdir()
    if nfc(p.name) in [
        'daily.csv',
        'rainfall_daily.csv'
    ]
]

sixh_candidates = [
    p for p in CLIMATE_DIR.iterdir()
    if nfc(p.name) in [
        '6h.csv',
        'rainfall_6h.csv'
    ]
]

print("daily 후보:", [p.name for p in daily_candidates])
print("6h 후보:", [p.name for p in sixh_candidates])

daily_path = daily_candidates[0]
sixh_path = sixh_candidates[0]

rain_daily = pd.read_csv(
    daily_path,
    dtype={
        'region_code': str,
        'TA_YMD': str
    }
)

rain_6h = pd.read_csv(
    sixh_path,
    dtype={
        'region_code': str,
        'TA_YMD': str
    }
)

rain_daily['date'] = pd.to_datetime(
    rain_daily['date']
)

rain_6h['date'] = pd.to_datetime(
    rain_6h['date']
)

print("daily shape:", rain_daily.shape)
print("6h shape:", rain_6h.shape)

daily 후보: ['rainfall_daily.csv']
6h 후보: ['rainfall_6h.csv']
daily shape: (368, 53)
6h shape: (1472, 20)


In [4]:
print("===== DAILY =====")

print(
    "기간:",
    rain_daily['date'].min(),
    "~",
    rain_daily['date'].max()
)

print("\n지역별 행 수:")
print(
    rain_daily['region']
    .value_counts()
)

print(
    "\n날짜×지역 중복:",
    rain_daily
    .duplicated(['date', 'region'])
    .sum()
)

print(
    "\n원본 precip_mm 결측:",
    rain_daily['precip_mm']
    .isna()
    .sum()
)

print(
    "보완 precip_mm_filled 결측:",
    rain_daily['precip_mm_filled']
    .isna()
    .sum()
)

print("\n보완 방법:")
print(
    rain_daily[
        'precip_fill_method'
    ].value_counts(dropna=False)
)


print("\n\n===== 6H =====")

print(
    "날짜×지역×시간대 중복:",
    rain_6h
    .duplicated(
        [
            'date',
            'region',
            'TIME_GB'
        ]
    )
    .sum()
)

print(
    "\n원본 precip_mm 결측:",
    rain_6h['precip_mm']
    .isna()
    .sum()
)

print(
    "보완 precip_mm_filled 결측:",
    rain_6h['precip_mm_filled']
    .isna()
    .sum()
)

print("\n보완 방법:")
print(
    rain_6h[
        'precip_fill_method'
    ].value_counts(dropna=False)
)

===== DAILY =====
기간: 2025-07-01 00:00:00 ~ 2025-12-31 00:00:00

지역별 행 수:
region
강원 춘천시    184
서울 강남구    184
Name: count, dtype: int64

날짜×지역 중복: 0

원본 precip_mm 결측: 4
보완 precip_mm_filled 결측: 0

보완 방법:
precip_fill_method
observed                           364
gap_filled:M3_mean_3_within_5km      4
Name: count, dtype: int64


===== 6H =====
날짜×지역×시간대 중복: 0

원본 precip_mm 결측: 12
보완 precip_mm_filled 결측: 0

보완 방법:
precip_fill_method
observed                           1460
gap_filled:M3_mean_3_within_5km      12
Name: count, dtype: int64


In [5]:
rain_summer = rain_daily[
    rain_daily['date'].between(
        '2025-07-01',
        '2025-09-30'
    )
].copy()

print("행 수:", len(rain_summer))
print("날짜 수:", rain_summer['date'].nunique())

print("\n지역별 날짜 수:")
print(
    rain_summer
    .groupby('region')['date']
    .nunique()
)

행 수: 184
날짜 수: 92

지역별 날짜 수:
region
강원 춘천시    92
서울 강남구    92
Name: date, dtype: int64


In [6]:
# 비가 조금이라도 온 날
rain_summer['rain_any'] = (
    rain_summer[
        'precip_mm_filled'
    ] > 0
).astype(int)


# 일강수 20mm 이상
rain_summer['rain_20'] = (
    rain_summer[
        'precip_mm_filled'
    ] >= 20
).astype(int)


# 일강수 50mm 이상
rain_summer['rain_50'] = (
    rain_summer[
        'precip_mm_filled'
    ] >= 50
).astype(int)


# 강수량 연속변수
rain_summer['rain_mm'] = (
    rain_summer[
        'precip_mm_filled'
    ]
)


# 비가 많이 오는 날의 큰 값 영향을 줄이기 위한 로그 강수량
rain_summer['log_rain'] = np.log1p(
    rain_summer[
        'precip_mm_filled'
    ]
)

In [7]:
rain_event_summary = (
    rain_summer
    .groupby('region')
    .agg(
        전체일수=(
            'date',
            'count'
        ),

        강수일수=(
            'rain_any',
            'sum'
        ),

        강수20mm이상=(
            'rain_20',
            'sum'
        ),

        강수50mm이상=(
            'rain_50',
            'sum'
        ),

        평균강수량=(
            'rain_mm',
            'mean'
        ),

        최대강수량=(
            'rain_mm',
            'max'
        )
    )
)

display(
    rain_event_summary.round(2)
)

,전체일수,강수일수,강수20mm이상,강수50mm이상,평균강수량,최대강수량
region,,,,,,
강원 춘천시,92,34,16,7,10.00,94.7
서울 강남구,92,36,11,3,7.74,112.5


In [8]:
gap_filled_days = rain_summer[
    rain_summer[
        'precip_fill_method'
    ] != 'observed'
][
    [
        'date',
        'region',
        'precip_mm',
        'precip_mm_filled',
        'precip_fill_method',
        'precip_fill_hours'
    ]
]

display(gap_filled_days)

,date,region,precip_mm,precip_mm_filled,precip_fill_method,precip_fill_hours
255,2025-09-10,서울 강남구,NaN,0.0,gap_filled:M3_mean_3_within_5km,16
256,2025-09-11,서울 강남구,NaN,0.0,gap_filled:M3_mean_3_within_5km,24
258,2025-09-13,서울 강남구,NaN,62.5,gap_filled:M3_mean_3_within_5km,14


In [9]:
# card_data.csv 위치 자동 탐색
card_candidates = list(
    BASE.rglob('card_data.csv')
)

print("card_data.csv 후보 수:", len(card_candidates))

for p in card_candidates:
    print(p)

card_data.csv 후보 수: 1
/content/drive/MyDrive/와빅_빅콘/card_data/card_data.csv


In [10]:
CARD_PATH = card_candidates[0]

usecols = [
    'TA_YMD',
    'TIME_GB',
    'MCT_SGG_CD',
    'MCT_RY_CD',
    'SEX_CCD',
    'AGE_CCD',
    'TS_AT',
    'USE_CNT'
]

card = pd.read_csv(
    CARD_PATH,
    usecols=usecols,
    dtype={
        'TA_YMD': str,
        'TIME_GB': str,
        'MCT_SGG_CD': str,
        'MCT_RY_CD': str,
        'SEX_CCD': str,
        'AGE_CCD': str
    }
)

card['date'] = pd.to_datetime(
    card['TA_YMD'],
    format='%Y%m%d'
)

print("전체 행 수:", len(card))
print("기간:", card['date'].min(), "~", card['date'].max())
print("지역:", card['MCT_SGG_CD'].unique())

전체 행 수: 1044710
기간: 2025-07-01 00:00:00 ~ 2025-12-31 00:00:00
지역: ['서울 강남구' '강원 춘천시']


In [11]:
consumer = card[
    (card['SEX_CCD'] != '법인')
    &
    (
        card['date'].between(
            '2025-07-01',
            '2025-09-30'
        )
    )
].copy()

print("행 수:", len(consumer))
print("날짜 수:", consumer['date'].nunique())
print("지역 수:", consumer['MCT_SGG_CD'].nunique())
print("업종 수:", consumer['MCT_RY_CD'].nunique())

행 수: 495841
날짜 수: 92
지역 수: 2
업종 수: 90


In [12]:
daily_card = (
    consumer
    .groupby(
        ['date', 'MCT_SGG_CD'],
        as_index=False
    )
    [['TS_AT', 'USE_CNT']]
    .sum()
    .rename(
        columns={
            'MCT_SGG_CD': 'region'
        }
    )
)

daily_card['avg_ticket'] = (
    daily_card['TS_AT']
    /
    daily_card['USE_CNT']
)

daily_card['log_sales'] = np.log1p(
    daily_card['TS_AT']
)

daily_card['log_count'] = np.log1p(
    daily_card['USE_CNT']
)

daily_card['log_ticket'] = np.log(
    daily_card['avg_ticket']
)

print("행 수:", len(daily_card))
print("\n지역별 날짜 수:")
print(
    daily_card
    .groupby('region')['date']
    .nunique()
)

display(daily_card.head())

행 수: 184

지역별 날짜 수:
region
강원 춘천시    92
서울 강남구    92
Name: date, dtype: int64


,date,region,TS_AT,USE_CNT,avg_ticket,log_sales,log_count,log_ticket
0,2025-07-01,강원 춘천시,9837406760,256297,38382.840064,23.009458,12.454096,10.555366
1,2025-07-01,서울 강남구,162641015798,3005058,54122.421530,25.814811,14.915808,10.899004
2,2025-07-02,강원 춘천시,8863857611,259602,34144.026668,22.905248,12.466909,10.438343
3,2025-07-02,서울 강남구,129717582617,2918879,44440.890704,25.588625,14.886711,10.701915
4,2025-07-03,강원 춘천시,7924855447,260455,30426.966067,22.793270,12.470189,10.323085


In [13]:
rain_for_merge = rain_summer[
    [
        'date',
        'region',
        'precip_mm',
        'precip_mm_filled',
        'precip_fill_method',
        'rain_any',
        'rain_20',
        'rain_50',
        'rain_mm',
        'log_rain'
    ]
].copy()

daily_rain = daily_card.merge(
    rain_for_merge,
    on=['date', 'region'],
    how='left',
    validate='one_to_one'
)

print("합친 행 수:", len(daily_rain))

print("\n강수변수 결측:")
print(
    daily_rain[
        [
            'precip_mm_filled',
            'rain_any',
            'rain_20'
        ]
    ]
    .isna()
    .sum()
)

합친 행 수: 184

강수변수 결측:
precip_mm_filled    0
rain_any            0
rain_20             0
dtype: int64


In [14]:
daily_rain['dow_num'] = (
    daily_rain['date'].dt.dayofweek
)

daily_rain['month'] = (
    daily_rain['date'].dt.month
)

daily_rain['trend'] = (
    daily_rain['date']
    -
    daily_rain['date'].min()
).dt.days


# 휴가기간
daily_rain['vacation_period'] = (
    daily_rain['date'].between(
        '2025-07-25',
        '2025-08-10'
    )
).astype(int)


# 소비쿠폰 지급 시작 이후
daily_rain['coupon_post'] = (
    daily_rain['date']
    >= pd.Timestamp('2025-07-21')
).astype(int)


# 분석기간 내 공휴일
daily_rain['holiday'] = (
    daily_rain['date']
    ==
    pd.Timestamp('2025-08-15')
).astype(int)

In [15]:
simple_rain_any = (
    daily_rain
    .groupby(
        ['region', 'rain_any']
    )
    .agg(
        일수=('date', 'count'),
        평균매출=('TS_AT', 'mean'),
        평균건수=('USE_CNT', 'mean'),
        평균객단가=('avg_ticket', 'mean')
    )
    .reset_index()
)

display(simple_rain_any)

,region,rain_any,일수,평균매출,평균건수,평균객단가
0,강원 춘천시,0,58,1.119203e+10,2.844233e+05,39125.617949
1,강원 춘천시,1,34,1.092815e+10,2.655260e+05,41340.531134
2,서울 강남구,0,56,1.618315e+11,2.908838e+06,54129.448586
3,서울 강남구,1,36,1.362995e+11,2.771406e+06,47990.056235


In [16]:
simple_rain20 = (
    daily_rain
    .groupby(
        ['region', 'rain_20']
    )
    .agg(
        일수=('date', 'count'),
        평균매출=('TS_AT', 'mean'),
        평균건수=('USE_CNT', 'mean'),
        평균객단가=('avg_ticket', 'mean')
    )
    .reset_index()
)

display(simple_rain20)

,region,rain_20,일수,평균매출,평균건수,평균객단가
0,강원 춘천시,0,76,1.106302e+10,2.807948e+05,39227.555374
1,강원 춘천시,1,16,1.124410e+10,2.615019e+05,43348.105698
2,서울 강남구,0,81,1.557912e+11,2.872952e+06,52686.148074
3,서울 강남구,1,11,1.227513e+11,2.723314e+06,44664.831936


In [17]:
def simple_change_table(df, shock):

    rows = []

    for region in df['region'].unique():

        temp = df[
            df['region'] == region
        ]

        normal = temp[
            temp[shock] == 0
        ]

        event = temp[
            temp[shock] == 1
        ]

        rows.append({

            'region': region,

            'event_days':
                len(event),

            'sales_change_pct':
                (
                    event['TS_AT'].mean()
                    /
                    normal['TS_AT'].mean()
                    - 1
                ) * 100,

            'count_change_pct':
                (
                    event['USE_CNT'].mean()
                    /
                    normal['USE_CNT'].mean()
                    - 1
                ) * 100,

            'ticket_change_pct':
                (
                    event['avg_ticket'].mean()
                    /
                    normal['avg_ticket'].mean()
                    - 1
                ) * 100
        })

    return pd.DataFrame(rows)


print("===== 비 여부 =====")
display(
    simple_change_table(
        daily_rain,
        'rain_any'
    ).round(2)
)

print("\n===== 20mm 이상 =====")
display(
    simple_change_table(
        daily_rain,
        'rain_20'
    ).round(2)
)

===== 비 여부 =====


,region,event_days,sales_change_pct,count_change_pct,ticket_change_pct
0,강원 춘천시,34,-2.36,-6.64,5.66
1,서울 강남구,36,-15.78,-4.72,-11.34



===== 20mm 이상 =====


,region,event_days,sales_change_pct,count_change_pct,ticket_change_pct
0,강원 춘천시,16,1.64,-6.87,10.50
1,서울 강남구,11,-21.21,-5.21,-15.22


In [18]:
# 강수구간을 명시적으로 다시 생성
daily_rain['rain_group'] = np.select(
    [
        daily_rain['precip_mm_filled'] == 0,

        (daily_rain['precip_mm_filled'] > 0) &
        (daily_rain['precip_mm_filled'] < 20),

        daily_rain['precip_mm_filled'] >= 20
    ],
    [
        '0mm_무강수',
        '0~20mm_약한강수',
        '20mm이상_강한강수'
    ],
    default='확인필요'
)

rain_group_count = (
    daily_rain
    .groupby(
        ['region', 'rain_group']
    )
    .size()
    .reset_index(name='일수')
)

display(rain_group_count)

,region,rain_group,일수
0,강원 춘천시,0mm_무강수,58
1,강원 춘천시,0~20mm_약한강수,18
2,강원 춘천시,20mm이상_강한강수,16
3,서울 강남구,0mm_무강수,56
4,서울 강남구,0~20mm_약한강수,25
5,서울 강남구,20mm이상_강한강수,11


In [19]:
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

In [20]:
rain_shocks = {
    'rain_any': '강수 여부',
    'rain_20': '20mm 이상 강수'
}

rain_reg_results = []

for region in [
    '강원 춘천시',
    '서울 강남구'
]:

    temp = (
        daily_rain[
            daily_rain['region'] == region
        ]
        .copy()
        .sort_values('date')
    )

    for shock, shock_name in rain_shocks.items():

        for outcome in [
            'log_sales',
            'log_count',
            'log_ticket'
        ]:

            formula = (
                f'{outcome} ~ '
                f'{shock}'
                ' + C(dow_num)'
                ' + C(month)'
                ' + trend'
                ' + vacation_period'
                ' + coupon_post'
                ' + holiday'
            )

            model = (
                smf.ols(
                    formula=formula,
                    data=temp
                )
                .fit(
                    cov_type='HAC',
                    cov_kwds={
                        'maxlags': 7
                    }
                )
            )

            beta = model.params[shock]

            ci = model.conf_int().loc[
                shock
            ]

            rain_reg_results.append({

                'region':
                    region,

                'shock':
                    shock_name,

                'outcome':
                    outcome,

                'effect_pct':
                    (
                        np.exp(beta) - 1
                    ) * 100,

                'ci95_lower_pct':
                    (
                        np.exp(ci.iloc[0]) - 1
                    ) * 100,

                'ci95_upper_pct':
                    (
                        np.exp(ci.iloc[1]) - 1
                    ) * 100,

                'p_value':
                    model.pvalues[shock],

                'event_days':
                    int(
                        temp[shock].sum()
                    ),

                'n':
                    int(model.nobs)
            })


rain_reg_results = pd.DataFrame(
    rain_reg_results
)

display(
    rain_reg_results.round(3)
)

,region,shock,outcome,effect_pct,ci95_lower_pct,ci95_upper_pct,p_value,event_days,n
0,강원 춘천시,강수 여부,log_sales,1.057,-10.863,14.570,0.870,34,92
1,강원 춘천시,강수 여부,log_count,-5.886,-7.818,-3.914,0.000,34,92
2,강원 춘천시,강수 여부,log_ticket,7.377,-4.800,21.112,0.246,34,92
3,강원 춘천시,20mm 이상 강수,log_sales,5.243,-13.203,27.610,0.603,16,92
4,강원 춘천시,20mm 이상 강수,log_count,-7.764,-9.564,-5.928,0.000,16,92
5,강원 춘천시,20mm 이상 강수,log_ticket,14.102,-4.904,36.907,0.156,16,92
6,서울 강남구,강수 여부,log_sales,-3.980,-10.233,2.709,0.237,36,92
7,서울 강남구,강수 여부,log_count,-2.807,-5.221,-0.332,0.026,36,92
8,서울 강남구,강수 여부,log_ticket,-1.206,-5.837,3.652,0.620,36,92
9,서울 강남구,20mm 이상 강수,log_sales,-8.241,-18.542,3.363,0.157,11,92


In [21]:
def run_rain_hac_models(df, spec_name):

    results = []

    rain_shocks = {
        'rain_any': '강수 여부',
        'rain_20': '20mm 이상 강수'
    }

    for region in [
        '강원 춘천시',
        '서울 강남구'
    ]:

        temp = (
            df[
                df['region'] == region
            ]
            .copy()
            .sort_values('date')
        )

        for shock, shock_name in rain_shocks.items():

            for outcome in [
                'log_sales',
                'log_count',
                'log_ticket'
            ]:

                formula = (
                    f'{outcome} ~ '
                    f'{shock}'
                    ' + C(dow_num)'
                    ' + C(month)'
                    ' + trend'
                    ' + vacation_period'
                    ' + coupon_post'
                    ' + holiday'
                )

                model = (
                    smf.ols(
                        formula=formula,
                        data=temp
                    )
                    .fit(
                        cov_type='HAC',
                        cov_kwds={
                            'maxlags': 7
                        }
                    )
                )

                beta = model.params[shock]

                ci = (
                    model
                    .conf_int()
                    .loc[shock]
                )

                results.append({

                    'spec':
                        spec_name,

                    'region':
                        region,

                    'shock':
                        shock_name,

                    'outcome':
                        outcome,

                    'effect_pct':
                        (
                            np.exp(beta) - 1
                        ) * 100,

                    'ci95_lower_pct':
                        (
                            np.exp(ci.iloc[0]) - 1
                        ) * 100,

                    'ci95_upper_pct':
                        (
                            np.exp(ci.iloc[1]) - 1
                        ) * 100,

                    'p_value':
                        model.pvalues[shock],

                    'event_days':
                        int(
                            temp[shock].sum()
                        ),

                    'n':
                        int(model.nobs)
                })

    return pd.DataFrame(results)

In [22]:
# ① 현재 주 분석: gap-filled 값 포함
result_main = run_rain_hac_models(
    daily_rain,
    '기본_보완값포함'
)


# ② 민감도 분석:
# gap-filled 날짜 자체를 분석에서 제외
daily_rain_observed = (
    daily_rain[
        daily_rain[
            'precip_fill_method'
        ] == 'observed'
    ]
    .copy()
)

result_observed = run_rain_hac_models(
    daily_rain_observed,
    '보완행제외'
)


rain_sensitivity = pd.concat(
    [
        result_main,
        result_observed
    ],
    ignore_index=True
)

display(
    rain_sensitivity.round(3)
)

,spec,region,shock,outcome,effect_pct,ci95_lower_pct,ci95_upper_pct,p_value,event_days,n
0,기본_보완값포함,강원 춘천시,강수 여부,log_sales,1.057,-10.863,14.570,0.870,34,92
1,기본_보완값포함,강원 춘천시,강수 여부,log_count,-5.886,-7.818,-3.914,0.000,34,92
2,기본_보완값포함,강원 춘천시,강수 여부,log_ticket,7.377,-4.800,21.112,0.246,34,92
3,기본_보완값포함,강원 춘천시,20mm 이상 강수,log_sales,5.243,-13.203,27.610,0.603,16,92
4,기본_보완값포함,강원 춘천시,20mm 이상 강수,log_count,-7.764,-9.564,-5.928,0.000,16,92
5,기본_보완값포함,강원 춘천시,20mm 이상 강수,log_ticket,14.102,-4.904,36.907,0.156,16,92
6,기본_보완값포함,서울 강남구,강수 여부,log_sales,-3.980,-10.233,2.709,0.237,36,92
7,기본_보완값포함,서울 강남구,강수 여부,log_count,-2.807,-5.221,-0.332,0.026,36,92
8,기본_보완값포함,서울 강남구,강수 여부,log_ticket,-1.206,-5.837,3.652,0.620,36,92
9,기본_보완값포함,서울 강남구,20mm 이상 강수,log_sales,-8.241,-18.542,3.363,0.157,11,92


In [23]:
effect_compare = (
    rain_sensitivity
    .pivot_table(
        index=[
            'region',
            'shock',
            'outcome'
        ],
        columns='spec',
        values='effect_pct'
    )
    .reset_index()
)

p_compare = (
    rain_sensitivity
    .pivot_table(
        index=[
            'region',
            'shock',
            'outcome'
        ],
        columns='spec',
        values='p_value'
    )
    .reset_index()
)

print("===== 효과 추정치 비교 (%) =====")
display(
    effect_compare.round(2)
)

print("\n===== p-value 비교 =====")
display(
    p_compare.round(4)
)

===== 효과 추정치 비교 (%) =====


spec,region,shock,outcome,기본_보완값포함,보완행제외
0,강원 춘천시,20mm 이상 강수,log_count,-7.76,-7.76
1,강원 춘천시,20mm 이상 강수,log_sales,5.24,5.24
2,강원 춘천시,20mm 이상 강수,log_ticket,14.10,14.10
3,강원 춘천시,강수 여부,log_count,-5.89,-5.89
4,강원 춘천시,강수 여부,log_sales,1.06,1.06
5,강원 춘천시,강수 여부,log_ticket,7.38,7.38
6,서울 강남구,20mm 이상 강수,log_count,-5.38,-5.57
7,서울 강남구,20mm 이상 강수,log_sales,-8.24,-8.38
8,서울 강남구,20mm 이상 강수,log_ticket,-3.03,-2.98
9,서울 강남구,강수 여부,log_count,-2.81,-2.74



===== p-value 비교 =====


spec,region,shock,outcome,기본_보완값포함,보완행제외
0,강원 춘천시,20mm 이상 강수,log_count,0.0000,0.0000
1,강원 춘천시,20mm 이상 강수,log_sales,0.6032,0.6032
2,강원 춘천시,20mm 이상 강수,log_ticket,0.1559,0.1559
3,강원 춘천시,강수 여부,log_count,0.0000,0.0000
4,강원 춘천시,강수 여부,log_sales,0.8696,0.8696
5,강원 춘천시,강수 여부,log_ticket,0.2464,0.2464
6,서울 강남구,20mm 이상 강수,log_count,0.0155,0.0169
7,서울 강남구,20mm 이상 강수,log_sales,0.1569,0.1733
8,서울 강남구,20mm 이상 강수,log_ticket,0.4609,0.4930
9,서울 강남구,강수 여부,log_count,0.0265,0.0216


In [24]:
rain_group_results = []

for region in [
    '강원 춘천시',
    '서울 강남구'
]:

    temp = (
        daily_rain[
            daily_rain['region'] == region
        ]
        .copy()
        .sort_values('date')
    )

    for outcome in [
        'log_sales',
        'log_count',
        'log_ticket'
    ]:

        formula = (
            f'{outcome} ~ '
            'C(rain_group, Treatment(reference="0mm_무강수"))'
            ' + C(dow_num)'
            ' + C(month)'
            ' + trend'
            ' + vacation_period'
            ' + coupon_post'
            ' + holiday'
        )

        model = (
            smf.ols(
                formula=formula,
                data=temp
            )
            .fit(
                cov_type='HAC',
                cov_kwds={
                    'maxlags': 7
                }
            )
        )

        for group_name in [
            '0~20mm_약한강수',
            '20mm이상_강한강수'
        ]:

            term = (
                'C(rain_group, Treatment(reference="0mm_무강수"))'
                f'[T.{group_name}]'
            )

            beta = model.params[term]
            ci = model.conf_int().loc[term]

            rain_group_results.append({

                'region':
                    region,

                'outcome':
                    outcome,

                'rain_group':
                    group_name,

                'effect_pct':
                    (np.exp(beta) - 1) * 100,

                'ci95_lower_pct':
                    (np.exp(ci.iloc[0]) - 1) * 100,

                'ci95_upper_pct':
                    (np.exp(ci.iloc[1]) - 1) * 100,

                'p_value':
                    model.pvalues[term],

                'n':
                    int(model.nobs)
            })


rain_group_results = pd.DataFrame(
    rain_group_results
)

display(
    rain_group_results.round(3)
)

,region,outcome,rain_group,effect_pct,ci95_lower_pct,ci95_upper_pct,p_value,n
0,강원 춘천시,log_sales,0~20mm_약한강수,-2.310,-13.555,10.397,0.708,92
1,강원 춘천시,log_sales,20mm이상_강한강수,4.718,-13.988,27.492,0.646,92
2,강원 춘천시,log_count,0~20mm_약한강수,-3.387,-5.715,-1.001,0.006,92
3,강원 춘천시,log_count,20mm이상_강한강수,-8.442,-10.328,-6.516,0.000,92
4,강원 춘천시,log_ticket,0~20mm_약한강수,1.115,-10.906,14.757,0.864,92
5,강원 춘천시,log_ticket,20mm이상_강한강수,14.373,-4.836,37.459,0.152,92
6,서울 강남구,log_sales,0~20mm_약한강수,-1.728,-7.772,4.712,0.590,92
7,서울 강남구,log_sales,20mm이상_강한강수,-8.661,-19.057,3.070,0.142,92
8,서울 강남구,log_count,0~20mm_약한강수,-1.420,-3.816,1.036,0.255,92
9,서울 강남구,log_count,20mm이상_강한강수,-5.734,-9.882,-1.394,0.010,92


In [25]:
rain_group_table = (
    rain_group_results[
        [
            'region',
            'outcome',
            'rain_group',
            'effect_pct',
            'p_value'
        ]
    ]
    .pivot(
        index=[
            'region',
            'outcome'
        ],
        columns='rain_group',
        values=[
            'effect_pct',
            'p_value'
        ]
    )
)

display(
    rain_group_table.round(3)
)

effect_pct                 p_value            
rain_group        0~20mm_약한강수 20mm이상_강한강수 0~20mm_약한강수 20mm이상_강한강수
region outcome                                                   
강원 춘천시 log_count       -3.387      -8.442       0.006       0.000
       log_sales       -2.310       4.718       0.708       0.646
       log_ticket       1.115      14.373       0.864       0.152
서울 강남구 log_count       -1.420      -5.734       0.255       0.010
       log_sales       -1.728      -8.661       0.590       0.142
       log_ticket      -0.313      -3.106       0.898       0.458